# Part C · Static embeddings + deep networks
`src/static_nets.py` holds the text pipeline, the three models and the training loop. This notebook runs the experiments, one cell per run.

```
tweet -> normalize -> words -> vocabulary (train, freq >= 2) -> embedding -> model -> sarcasm logit

C1  BiLSTM                       C2  BiLSTM + attention             C3  Mini Transformer
embedding (100)                  embedding (100)                    embedding (100) -> Linear 128
BiLSTM (2 x 128)                 BiLSTM (2 x 128)                   + sinusoidal positions (ablation: none)
masked max-pooling               attention pooling (own code)       2 x [4-head self-attention + feed-forward]
Linear -> 2                      Linear -> 2                        masked mean-pooling -> Linear -> 2
```

Three embeddings for every model: **random**, **AraVec frozen**, **AraVec unfrozen** (AraVec Twitter, unigrams, 100 dimensions, `clean_str` applied before the lookup).

| Stage | Runs | Data used |
|---|---|---|
| 1. 3 models × 3 embeddings, weighted loss | 9 | validation only |
| 2. C3 without positional encoding | 1 | validation only |
| 3. Imbalance on the best model: oversampling, standard loss + threshold (weighted is already stage 1) | 2 | validation only |
| 4. Best configuration, 3 seeds | 3 | validation, then the test once |

The training budget is the same for every model (see the top of `static_nets.py`): early stopping on validation F1 of the sarcastic class. The test set is only touched in stage 4. Stage 4 saves what Part F needs: test predictions for 3 seeds and one checkpoint with its vocabulary and threshold.

## 1. Setup

In [ ]:
import os
import sys

REPO = "Arabic-sarcasm"

if not os.path.exists(f"/content/{REPO}"):
    !git clone https://github.com/cad0m/{REPO}.git /content/{REPO}
else:
    !git -C /content/{REPO} pull

%cd /content/{REPO}

if f"/content/{REPO}" not in sys.path:
    sys.path.insert(0, f"/content/{REPO}")

!pip install -q uv
!uv pip install --system emoji tashaphyne

Results, predictions and checkpoints are written straight to Google Drive, so a disconnected session loses nothing.

In [ ]:
import shutil
from pathlib import Path

ROOT = Path("/content/Arabic-sarcasm")
DRIVE = Path("/content/drive/MyDrive/sarcasm")

for name in ["results", "predictions", "models"]:
    local, remote = ROOT / name, DRIVE / name
    if not remote.exists():                       # first time: copy what the repo already has
        if local.exists():
            shutil.copytree(local, remote)
        else:
            remote.mkdir(parents=True)
    if local.exists() and not local.is_symlink():
        shutil.rmtree(local)
    if not local.exists():
        local.symlink_to(remote)

## 2. Data

In [ ]:
import sys
sys.path.insert(0, "/content/Arabic-sarcasm")

import numpy as np
import pandas as pd

from src.data import get_data
from src.evaluate import RESULTS, log_result, metrics, plot_results, save_predictions, show
from src.static_nets import (build_model, build_vocab, class_weights, encode, load_aravec, oov_rate,
                             predict_proba, save_checkpoint, tokenize, train_model)
from src.train import best_threshold, count_params, oversample, set_seed

PREP = "static_normalize"       # label written in results.csv
ARAVEC_PATH = DRIVE / "aravec" / "full_uni_cbow_100_twitter.mdl"

train, val, test, report = get_data()
y_val = val["sarcasm"].to_numpy()
y_test = test["sarcasm"].to_numpy()
print(report)

## 3. Words, vocabulary and OOV
The tweet is normalized (URL and mention placeholders, no diacritics or tatweel, unified letters), then split into words. Punctuation marks and emojis are kept as separate tokens because they can be sarcasm cues. The vocabulary comes from train only.

In [ ]:
tokens_train = [tokenize(t) for t in train["tweet"]]
tokens_val = [tokenize(t) for t in val["tweet"]]
tokens_test = [tokenize(t) for t in test["tweet"]]

vocab = build_vocab(tokens_train)
print("vocabulary size:", len(vocab))
print(f"OOV rate: val {oov_rate(tokens_val, vocab):.1%} | test {oov_rate(tokens_test, vocab):.1%}")

for tweet, tokens in zip(train["tweet"].head(3), tokens_train[:3]):
    print(tweet)
    print(tokens)
    print()

In [ ]:
def encode_all(texts):
    return [encode(tokenize(t), vocab) for t in texts]


x_val = encode_all(val["tweet"])
x_test = encode_all(test["tweet"])

## 4. AraVec
Download AraVec 3.0, Twitter, unigrams, CBOW, 100 dimensions from the AraVec GitHub page once, and put the `.mdl` file with its `.npy` files in `sarcasm/aravec` on Drive. Every word of the vocabulary goes through `clean_str` before the lookup; words not found keep a small random vector.

In [ ]:
assert ARAVEC_PATH.exists(), "put the AraVec files in sarcasm/aravec on Drive"

aravec_matrix, coverage = load_aravec(vocab, ARAVEC_PATH)
print(f"vocabulary words found in AraVec: {coverage:.1%}")

## 5. What one run does

In [ ]:
def run_one(arch, emb, imbalance, seed, positions=True, final=False):
    # arch: bilstm | bilstm_att | transformer      emb: random | aravec_frozen | aravec_unfrozen
    # imbalance: weighted | oversample | standard   final: score the test and save the files for Part F
    name = f"{arch}_{emb}_{imbalance}" + ("" if positions else "_nopos")

    # 1. data of this run (oversampling happens on train only)
    train_df = oversample(train, seed) if imbalance == "oversample" else train
    x_tr = encode_all(train_df["tweet"])
    y_tr = train_df["sarcasm"].to_numpy()
    weights = class_weights(y_tr) if imbalance == "weighted" else None

    # 2. train (the model is left at its best epoch)
    set_seed(seed)
    matrix = None if emb == "random" else aravec_matrix
    model = build_model(arch, len(vocab), matrix, freeze=(emb == "aravec_frozen"), positions=positions)
    train_time = train_model(model, x_tr, y_tr, x_val, y_val, weights)

    # 3. validation: choose the threshold
    val_proba = predict_proba(model, x_val)
    threshold = best_threshold(y_val, val_proba)
    val_scores = metrics(y_val, val_proba, threshold)

    row = {"model": name, "preprocessing": PREP, "seed": seed, "arch": arch, "embedding": emb,
           "imbalance": imbalance, "positions": positions, "threshold": threshold,
           "train_time": train_time, "trained_params": count_params(model)}
    row.update({"val_" + k: v for k, v in val_scores.items()})

    # 4. final runs only: test with the validation threshold, predictions, checkpoint
    if final:
        test_proba = predict_proba(model, x_test)
        test_scores = metrics(y_test, test_proba, threshold)
        row.update({"test_" + k: v for k, v in test_scores.items()})
        save_predictions(name, seed, test["id"], y_test, test_proba, threshold)
        if seed == 42:
            save_checkpoint(ROOT / "models" / f"{name}_seed{seed}.pt", model, vocab, arch, positions, threshold)

    # 5. log
    log_result(**row)
    show(name, val_scores, threshold, train_time)


def load_results():
    table = pd.read_csv(RESULTS)
    return table[table["preprocessing"] == PREP]

## 6. Stage 1 · 3 models × 3 embeddings
Weighted loss, seed 42, validation only.

### C1 · BiLSTM

In [ ]:
run_one("bilstm", "random", "weighted", 42)

In [ ]:
run_one("bilstm", "aravec_frozen", "weighted", 42)

In [ ]:
run_one("bilstm", "aravec_unfrozen", "weighted", 42)

### C2 · BiLSTM + attention

In [ ]:
run_one("bilstm_att", "random", "weighted", 42)

In [ ]:
run_one("bilstm_att", "aravec_frozen", "weighted", 42)

In [ ]:
run_one("bilstm_att", "aravec_unfrozen", "weighted", 42)

### C3 · Mini Transformer

In [ ]:
run_one("transformer", "random", "weighted", 42)

In [ ]:
run_one("transformer", "aravec_frozen", "weighted", 42)

In [ ]:
run_one("transformer", "aravec_unfrozen", "weighted", 42)

Validation results of stage 1.

In [ ]:
results = load_results()
stage1 = results[(results["imbalance"] == "weighted") & results["positions"] & (results["seed"] == 42)]
stage1[["arch", "embedding", "val_f1", "val_pr_auc", "train_time", "trained_params"]].sort_values("val_f1", ascending=False)

## 7. Stage 2 · C3 without positional encoding
Same embedding as the best C3.

In [ ]:
c3 = stage1[stage1["arch"] == "transformer"].sort_values("val_f1")
best_emb_c3 = c3.iloc[-1]["embedding"]
print("best C3 embedding:", best_emb_c3)

In [ ]:
run_one("transformer", best_emb_c3, "weighted", 42, positions=False)

In [ ]:
results = load_results()
ablation = results[(results["arch"] == "transformer") & (results["embedding"] == best_emb_c3)
                   & (results["imbalance"] == "weighted") & (results["seed"] == 42)]
ablation[["model", "positions", "val_f1", "val_pr_auc"]]

## 8. Stage 3 · imbalance on the best model of C1–C3
(a) weighted loss is already in stage 1. Remaining: (b) oversampling and (c) standard loss with the threshold chosen on validation.

In [ ]:
best = stage1.sort_values("val_f1").iloc[-1]
best_arch, best_emb = best["arch"], best["embedding"]
print("best model of C1-C3:", best_arch, "+", best_emb)

In [ ]:
run_one(best_arch, best_emb, "oversample", 42)

In [ ]:
run_one(best_arch, best_emb, "standard", 42)

In [ ]:
results = load_results()
imbalance = results[(results["arch"] == best_arch) & (results["embedding"] == best_emb)
                    & results["positions"] & (results["seed"] == 42)]
imbalance[["imbalance", "val_f1", "val_pr_auc", "val_precision", "val_recall", "threshold"]]

## 9. Stage 4 · final model, 3 seeds
The configuration with the best validation F1 over all the runs above. These runs also score the test (with the validation threshold), save the predictions, and seed 42 saves the checkpoint.

In [ ]:
selection = results[results["positions"] & (results["seed"] == 42)]
final = selection.sort_values("val_f1").iloc[-1]
final_arch, final_emb, final_imb = final["arch"], final["embedding"], final["imbalance"]
print("final model:", final_arch, "+", final_emb, "+", final_imb)

In [ ]:
run_one(final_arch, final_emb, final_imb, 42, final=True)

In [ ]:
run_one(final_arch, final_emb, final_imb, 43, final=True)

In [ ]:
run_one(final_arch, final_emb, final_imb, 44, final=True)

## 10. Results

In [ ]:
results = load_results()
final_name = f"{final_arch}_{final_emb}_{final_imb}"
results[results["model"] == final_name].groupby("model")[["val_f1", "test_f1"]].agg(["mean", "std", "count"])

In [ ]:
plot_table = results[results["seed"] == 42].set_index("model")[["val_f1", "val_pr_auc"]]
plot_table.columns = ["f1", "pr_auc"]
plot_results(plot_table)